# Milestone M3–M4: Preprocessing Sentral & Pipeline Stratified Split
## Kelompok 4 — CNN for Text Classification (Indonesian Hate Speech Detection)

**Mata Kuliah:** Workshop Proyek Sistem Cerdas 2026  
**Dataset:** indotoxic2024  

### Tujuan Notebook:
1. Melakukan pembersihan teks (*cleaning*) menggunakan class `TextCleaner` (hapus URL, mention, tanda baca, lowercase).
2. Membagi data menjadi subset Train (70%), Validation (15%), dan Test (15%) menggunakan stratified split `DataSplitter`.
3. Membangun representasi vocabulary dan sequence integer dengan `TextTokenizer` (**fit HANYA pada data train** untuk mencegah data leakage).
4. Melakukan padding dan truncating sekuens menggunakan `SequencePadder`.
5. Menyimpan artefak dataset terproses (`data/processed/`, `data/splits/`) dan serializer tokenizer.

In [ ]:
import sys
from pathlib import Path
import os

# Tambahkan root project ke sys.path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import pandas as pd
import numpy as np

from src.utils.config import Config
from src.utils.seed import set_seed
from src.preprocessing.cleaner import TextCleaner
from src.preprocessing.splitter import DataSplitter
from src.preprocessing.tokenizer import TextTokenizer
from src.preprocessing.padder import SequencePadder

set_seed(Config.SEED)
print(f"Project root: {ROOT_DIR}")
print(f"Config MAX_LEN: {Config.MAX_LEN}, VOCAB_SIZE: {Config.VOCAB_SIZE}")

### 1. Pemuatan Data Mentah

In [ ]:
raw_path = ROOT_DIR / Config.RAW_DATA_CSV
if os.path.exists(raw_path):
    df = pd.read_csv(raw_path)
else:
    print("Membuat data simulasi untuk keperluan verifikasi pipeline...")
    df = pd.DataFrame({
        "text": [
            "Cek link http://example.com @user1 Dasar bajingan provokator! #rusuh",
            "Selamat pagi semuanya, semoga hari ini penuh sukacita dan damai.",
            "Hajar aja tuh penipu @penjahat! Gak usah dikasih ampun!",
            "Terima kasih atas ilmu dan diskusinya yang sangat bermanfaat.",
            "Dasar kelompok bodoh gak punya otak http://spam.xyz",
            "Mari kita jaga kerukunan antar umat beragama di Indonesia.",
            "Mampus lu kadrun cebong tolol perusak negeri @netizen",
            "Informasi beasiswa kuliah luar negeri sudah dibuka, silakan cek website resmi."
        ],
        "label": [1, 0, 1, 0, 1, 0, 1, 0]
    })

print(f"Jumlah data mentah: {len(df)}")
df.head()

### 2. Text Cleaning dengan `TextCleaner`

In [ ]:
cleaner = TextCleaner(
    remove_urls=True,
    remove_mentions=True,
    remove_hashtags=True,
    remove_punctuation=True,
    lowercase=True
)

df["text_clean"] = cleaner.clean_batch(df["text"].astype(str).tolist())

print("Sampel sebelum vs sesudah pembersihan:")
for i in range(min(4, len(df))):
    print(f"Raw   : {df['text'].iloc[i]}")
    print(f"Clean : {df['text_clean'].iloc[i]}")
    print("-" * 50)

# Simpan dataset hasil pembersihan
os.makedirs(ROOT_DIR / "data/processed", exist_ok=True)
df.to_csv(ROOT_DIR / Config.PROCESSED_DATA_PATH, index=False)
print(f"Data cleaned disimpan ke: {Config.PROCESSED_DATA_PATH}")

### 3. Stratified Data Splitting (70% Train, 15% Val, 15% Test) dengan `DataSplitter`

In [ ]:
splitter = DataSplitter(
    train_ratio=Config.TRAIN_RATIO,
    val_ratio=Config.VAL_RATIO,
    test_ratio=Config.TEST_RATIO,
    seed=Config.SEED
)

train_df, val_df, test_df = splitter.split(df, stratify_col="label")
saved_paths = splitter.save_splits(
    train_df, val_df, test_df, output_dir=str(ROOT_DIR / Config.SPLITS_DIR)
)

print(f"Train set: {len(train_df)} baris | Distribusi label:\n{train_df['label'].value_counts(normalize=True)}")
print(f"Val set  : {len(val_df)} baris | Distribusi label:\n{val_df['label'].value_counts(normalize=True)}")
print(f"Test set : {len(test_df)} baris | Distribusi label:\n{test_df['label'].value_counts(normalize=True)}")
print(f"Splits saved to: {saved_paths}")

### 4. Tokenisasi & Padding (`TextTokenizer` & `SequencePadder`)
**Catatan Krusial:** `tokenizer.fit()` hanya dipanggil pada `train_df['text_clean']` demi mencegah kebocoran informasi (*data leakage*) dari validation dan test set.

In [ ]:
tokenizer = TextTokenizer(
    vocab_size=Config.VOCAB_SIZE,
    oov_token=Config.OOV_TOKEN,
    pad_token=Config.PAD_TOKEN
)

# Fit HANYA pada train
tokenizer.fit(train_df["text_clean"].tolist())

# Transform semua subset
train_seqs = tokenizer.texts_to_sequences(train_df["text_clean"].tolist())
val_seqs = tokenizer.texts_to_sequences(val_df["text_clean"].tolist())
test_seqs = tokenizer.texts_to_sequences(test_df["text_clean"].tolist())

padder = SequencePadder(
    max_len=Config.MAX_LEN,
    padding="post",
    truncating="post"
)

X_train = padder.pad(train_seqs)
X_val = padder.pad(val_seqs)
X_test = padder.pad(test_seqs)

print(f"Bentuk X_train matrix: {X_train.shape}")
print(f"Bentuk X_val matrix  : {X_val.shape}")
print(f"Bentuk X_test matrix : {X_test.shape}")

# Simpan tokenizer untuk inferensi Streamlit & model testing
os.makedirs(ROOT_DIR / "outputs/tokenizer", exist_ok=True)
tokenizer.save(str(ROOT_DIR / Config.TOKENIZER_OUTPUT))
print(f"Tokenizer state berhasil disimpan ke: {Config.TOKENIZER_OUTPUT}")

### 5. Kesimpulan Milestone M3–M4
1. Text cleaning berhasil menstandardisasi input teks.
2. Partisi data 70/15/15 tersimpan rapi dalam format CSV.
3. Tokenizer dan Padder siap menyuplai representasi matrix integer untuk eksperimen baseline (M5) dan CNN (M6-M7).